# Лабораторна робота №2
## Наука про дані: підготовчий етап
### Налаштування віртуального середовища та інсталяція бібліотек

In [16]:
# Створення віртуального середовища у терміналі (виконується одноразово)
# python -m venv venv
# Активація середовища:
# Windows: venv\Scripts\activate
# Linux/MacOS: source venv/bin/activate

# Інсталяція необхідних пакетів:
# pip install pandas requests

### Імпорт модулів та константні дані

In [17]:
import os
import re
import urllib.request
from datetime import datetime
import pandas as pd

NOAA_TO_UKR = {
    1: {"name": "Вінницька", "ukr_id": 1},
    2: {"name": "Волинська", "ukr_id": 2},
    3: {"name": "Дніпропетровська", "ukr_id": 3},
    4: {"name": "Донецька", "ukr_id": 4},
    5: {"name": "Житомирська", "ukr_id": 5},
    6: {"name": "Закарпатська", "ukr_id": 6},
    7: {"name": "Запорізька", "ukr_id": 7},
    8: {"name": "Івано-Франківська", "ukr_id": 8},
    9: {"name": "Київська", "ukr_id": 9},
    10: {"name": "Кіровоградська", "ukr_id": 10},
    11: {"name": "Луганська", "ukr_id": 11},
    12: {"name": "Львівська", "ukr_id": 12},
    13: {"name": "Миколаївська", "ukr_id": 13},
    14: {"name": "Одеська", "ukr_id": 14},
    15: {"name": "Полтавська", "ukr_id": 15},
    16: {"name": "Рівненська", "ukr_id": 16},
    17: {"name": "Сумська", "ukr_id": 17},
    18: {"name": "Тернопільська", "ukr_id": 18},
    19: {"name": "Харківська", "ukr_id": 19},
    20: {"name": "Херсонська", "ukr_id": 20},
    21: {"name": "Хмельницька", "ukr_id": 21},
    22: {"name": "Черкаська", "ukr_id": 22},
    23: {"name": "Чернівецька", "ukr_id": 23},
    24: {"name": "Чернігівська", "ukr_id": 24},
    25: {"name": "Крим", "ukr_id": 25},
    26: {"name": "Київ", "ukr_id": 26},
    27: {"name": "Севастополь", "ukr_id": 27}
}

DOWNLOAD_DIR = "vhi_data"

### Завдання: 
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [18]:
def download_vhi_data(directory=DOWNLOAD_DIR):
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Створено директорію: {directory}")
    
    existing_files = os.listdir(directory)
    if existing_files:
        print("Директорія не порожня. Запобігання повторному довантаженню активовано. Завантаження скасовано.")
        return

    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean"
    
    for pid in range(1, 28):
        url = base_url.format(pid=pid)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"vhi_id_{pid}_{timestamp}.txt"
        filepath = os.path.join(directory, filename)
        
        try:
            print(f"Завантаження з підключенням до ID {pid}...")
            urllib.request.urlretrieve(url, filepath)
        except Exception as e:
            print(f"Помилка завантаження для ID {pid}: {e}")
            
download_vhi_data()

Директорія не порожня. Запобігання повторному довантаженню активовано. Завантаження скасовано.


### Завдання: 
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області. Реалізувати процедуру зміни індексів так, щоб області індексувалися за українською абеткою.

In [19]:
def load_and_clean_data(directory=DOWNLOAD_DIR):
    all_dfs = []
    if not os.path.exists(directory):
        print(f"Директорія {directory} відсутня.")
        return pd.DataFrame()
        
    files = [f for f in os.listdir(directory) if f.startswith("vhi_id_") and f.endswith(".txt")]
    
    if not files:
        print("Файли для зчитування відсутні у папці.")
        return pd.DataFrame()

    for filename in files:
        match = re.search(r"vhi_id_(\d+)_", filename)
        if not match:
            continue
        noaa_id = int(match.group(1))
        
        filepath = os.path.join(directory, filename)
        
        with open(filepath, 'r', encoding='utf-8') as file:
            content = file.read()
            
        # Якщо сервер повернув html-помилку замість даних
        if "<html" in content.lower() or "<pre" in content.lower():
            content = re.sub(r'<[^>]+>', '', content)
            
        lines = content.split('\n')
        data_lines = []
        
        for line in lines:
            line_str = line.strip()
            # Перевірка: рядок повинен починатися з цифр (року)
            if line_str and re.match(r'^\s*\d{4}', line_str):
                data_lines.append(line_str)
        
        if not data_lines:
            continue
            
        # Парсинг з урахуванням того, що розділювачем може бути кома або пробіли
        split_data = []
        for line in data_lines:
            parts = [p.strip() for p in line.split(',') if p.strip()]
            if len(parts) >= 7:
                split_data.append(parts[:7])
        
        if not split_data:
            continue
            
        df = pd.DataFrame(split_data)
        df.columns = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
        df = df.dropna(subset=['Year', 'Week', 'VHI'])
        df = df[df['VHI'] >= 0] # Видаляємо значення -1
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        
        if noaa_id in NOAA_TO_UKR:
            df['Region_Name'] = NOAA_TO_UKR[noaa_id]['name']
            df['Region_ID'] = NOAA_TO_UKR[noaa_id]['ukr_id']
        else:
            df['Region_Name'] = "Unknown"
            df['Region_ID'] = noaa_id
            
        all_dfs.append(df)
        
    if not all_dfs:
        print("Не вдалося розпарсити жодного файлу. Перевірте вміст папки vhi_data.")
        return pd.DataFrame()
        
    final_df = pd.concat(all_dfs, ignore_index=True)
    return final_df

main_df = load_and_clean_data()
print(f"Загальна кількість записів після очищення: {len(main_df)}")

Загальна кількість записів після очищення: 59022


### Завдання: 
Реалізувати процедуру для формування вибірки: Ряд VHI для області за вказаний рік.

In [20]:
def get_vhi_series_by_year(df, region_id, year):
    if df.empty or 'Region_ID' not in df.columns:
        return pd.DataFrame()
    result = df[(df['Region_ID'] == region_id) & (df['Year'] == year)][['Week', 'VHI']].sort_values('Week')
    return result

# Приклад виклику для Вінницької області (ID: 1) за 2020 рік
print("Ряд VHI для Вінницької області за 2020 рік:")
print(get_vhi_series_by_year(main_df, 1, 2020).to_string(index=False))

Ряд VHI для Вінницької області за 2020 рік:
 Week   VHI
    1 37.80
    2 39.94
    3 41.75
    4 42.76
    5 42.19
    6 41.29
    7 40.53
    8 39.70
    9 39.11
   10 38.47
   11 37.11
   12 37.02
   13 36.29
   14 34.52
   15 32.40
   16 31.62
   17 34.03
   18 38.45
   19 45.09
   20 48.94
   21 49.44
   22 48.53
   23 49.55
   24 52.84
   25 56.61
   26 58.82
   27 60.26
   28 60.90
   29 60.08
   30 57.85
   31 54.05
   32 49.74
   33 44.30
   34 39.28
   35 35.38
   36 32.57
   37 31.92
   38 30.77
   39 29.64
   40 29.83
   41 30.55
   42 29.29
   43 29.36
   44 29.75
   45 32.12
   46 34.79
   47 38.60
   48 41.92
   49 43.23
   50 43.46
   51 42.98
   52 44.21


### Завдання: 
Реалізувати процедуру для формування вибірки: Ряд VHI за вказаний діапазон років для вказаних областей.

In [ ]:
def get_vhi_by_regions_and_years(df, region_ids, start_year, end_year):
    if df.empty or 'Region_ID' not in df.columns:
        return pd.DataFrame()
    result = df[
        (df['Region_ID'].isin(region_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ][['Region_ID', 'Region_Name', 'Year', 'Week', 'VHI']].sort_values(['Region_ID', 'Year', 'Week'])
    return result

# Приклад виклику для областей з ID 1 та 2 за період 2021-2022 років
print("Ряд VHI за 2021-2022 роки для областей ID [1, 2]:")
print(get_vhi_by_regions_and_years(main_df, [1, 2], 2021, 2022).to_string(index=False)) 

Ряд VHI за 2021-2022 роки для областей ID [1, 2]:
 Region_ID Region_Name  Year  Week   VHI
         1   Вінницька  2021     1 45.90
         1   Вінницька  2021     2 49.54
         1   Вінницька  2021     3 55.83
         1   Вінницька  2021     4 59.30
         1   Вінницька  2021     5 57.32
         1   Вінницька  2021     6 54.15
         1   Вінницька  2021     7 51.98
         1   Вінницька  2021     8 49.56
         1   Вінницька  2021     9 47.69
         1   Вінницька  2021    10 46.68
         1   Вінницька  2021    11 47.25
         1   Вінницька  2021    12 46.20
         1   Вінницька  2021    13 45.98
         1   Вінницька  2021    14 44.69
         1   Вінницька  2021    15 44.38
         1   Вінницька  2021    16 45.96
         1   Вінницька  2021    17 49.11
         1   Вінницька  2021    18 52.52
         1   Вінницька  2021    19 54.64
         1   Вінницька  2021    20 54.12
         1   Вінницька  2021    21 54.27
         1   Вінницька  2021    22 54.41
       

### Завдання: 
Реалізувати процедуру пошуку екстремумів (min та max) для вказаних областей та років, середнього, медіани.


In [22]:
def get_vhi_statistics(df, region_ids, start_year, end_year):
    if df.empty or 'Region_ID' not in df.columns:
        return "Немає даних для розрахунку або структура DataFrame некоректна."
        
    filtered_df = df[
        (df['Region_ID'].isin(region_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    
    if filtered_df.empty:
        return "Немає даних для розрахунку статистичних показників за вказаний період."
        
    stats = filtered_df.groupby(['Region_ID', 'Region_Name', 'Year'])['VHI'].agg(
        Мінімум='min',
        Максимум='max',
        Середнє='mean',
        Медіана='median'
    ).reset_index()
    
    return stats

# Приклад виклику для Вінницької області (ID: 1) за період 2019-2021 років
print("Статистичні показники VHI для області ID 1 за 2019-2021 роки:")
result = get_vhi_statistics(main_df, [1], 2019, 2021)

if isinstance(result, pd.DataFrame):
    display(result)
else:
    print(result)

Статистичні показники VHI для області ID 1 за 2019-2021 роки:


,Region_ID,Region_Name,Year,Мінімум,Максимум,Середнє,Медіана
0,1,Вінницька,2019,18.02,69.20,43.484231,40.93
1,1,Вінницька,2020,29.29,60.90,41.377500,39.82
2,1,Вінницька,2021,37.80,70.79,53.851923,53.46
